### Goal: Create Evals For A RAG vs. Graph RAG Agent

What I think I need v1:
1. Create a dataset (COMPLETE)
2. Take dataset and extract data
3. NER for game-specific entities
4. Perform relation extraction to find conextion between entities
5. Design graph schema and relationship types
6. Setup Neo4j
7. Create and configure Neo4j
8. Create embeddings
9. Implement community detection
10. Setup RAG
11. Create evaluation test set

Other things I need:
1. Add key links that are not in the sitemap (ex. abilities and I think trinkets aren't there for some reason)
2. No NER and relation extraction in the sense I was thinking of. Instead create a class/classes for the schema. I have to create the ontology myself
3. Use a class to create the ontology
4. In the class, also make it have the source file so when we get to the RAG step, we can just use the raw HTML for comparison
5. Once we get this complete, then it's learning step 6 & beyond

# Module Imports

In [ ]:
import asyncio
from asyncio import Semaphore
from bs4 import BeautifulSoup, Comment
import aiohttp
import requests

In [ ]:
from utils.sitemap_scraper import scrape_sitemap, process_sitemap_urls, scrape_multiple_links, save_raw_html_outputs
from utils.utils import generate_directories

In [ ]:
# Generate directories needed for project
# TODO: Get rid of this, generate the directories in the function itself
generate_directories()

### Scrape Sitemap & Save Locally
This will be used to embed the raw HTMLs

#### TODO: 
1. Make an orchestration function for this 
2. Save sitemap locally so I don't have to keep rerunning and scraping stuff I already have (for now)
3. Add missing links that I know we somehow missed in the site map

In [ ]:
# sitemap_url = 'https://wildfrostwiki.com/sitemap.xml'
# sitemap_urls = scrape_sitemap(sitemap_url)

In [ ]:
# urls = process_sitemap_urls(sitemap_urls)

In [ ]:
# html_outputs = await scrape_multiple_links(urls)

In [ ]:
# raw_html_subdirectory = 'raw_htmls'
# save_raw_html_outputs(html_outputs, raw_html_subdirectory)

### Scrape and save the sites in accordance to the ontology

Schemas:
1. Cards Schema: https://wildfrostwiki.com/index.php?title=Baby_Snowbo; grab the table at the end, create a folder structure based on that
2. Fights & Boss Battles: https://wildfrostwiki.com/The_Bog_Berries; grab the Map Events Table at the end
3. Charms: https://wildfrostwiki.com/Charms; another table
4. Stats, Buffs, Debuffs: https://wildfrostwiki.com/Stats
5. Keywords: https://wildfrostwiki.com/Keywords; relations between stats

In [ ]:
from utils.cards import CardType, CardInfo

In [ ]:
from utils.generate_schemas import generate_card_type_html_schema

In [ ]:
card_type_schema = generate_card_type_html_schema()

In [ ]:
import os
import json

filename = 'data/schemas/'
schema_filename = os.path.join(filename,'card_type_schema.json')
os.makedirs(filename, exist_ok=True)

with open(schema_filename,'w',encoding='utf-8') as f:
    json.dump(card_type_schema, f, indent=4)

In [ ]:
for k, v in card_type_schema.items():
    print(f'{k}: {v}')

In [ ]:
base_url = 'https://wildfrostwiki.com'

In [ ]:
import re

def clean_name_for_url(name: str) -> str:
    """Clean card name for use in URLs by replacing spaces with underscores"""
    return re.sub(r'\s+', '_', name)

In [ ]:
# Need to make sure that the sub_directory is part of the link. Might just use a tuple and use tuple unpacking into the scrape multiple links function
# TODO: 
#   1. Create a dictionary where each card link is placed in a subdirectory as a key 
#   2. Take the dictionary, scrape 

card_infos  = []
for card_type, cards in card_type_schema.items():
    if card_type == 'leaders':
        continue

    for card_name in cards:
        cleaned_name = clean_name_for_url(card_name)
        card_info = CardInfo(
            card_name=card_name,
            card_type=CardType(card_type),
            card_url=f'{base_url}/{cleaned_name}'
        )
        card_infos.append(card_info)

for c in card_infos:
    print(f'{c.card_name} {c.card_url}\n')

In [ ]:
urls = [card.card_url for card in card_infos]

In [ ]:
card_types_html_outputs = await scrape_multiple_links(urls, max_concurrent=100)

In [ ]:
for card_info, html in zip(card_infos, card_types_html_outputs):
    card_info.card_html = html
    if card_info.card_html is not None:
        card_info.save_html()
        card_info.parse_html()

In [ ]:
card_infos

In [ ]:
for c in card_infos:
    print(f'{c}\n')

In [ ]:
for c in card_infos:
    print(f'{c.to_dict()}\n')

### Leaders HTML

In [ ]:
# import requests
# from bs4 import BeautifulSoup

# leaders_url = 'https://wildfrostwiki.com/Leaders'
# response = requests.get(leaders_url)
# response.raise_for_status()

# soup = BeautifulSoup(response.text, 'html.parser')

# print(soup.prettify())

### Tribe Exclusivity Check

In [ ]:
import requests
from bs4 import BeautifulSoup

leaders_url = 'https://wildfrostwiki.com/Companions'
response = requests.get(leaders_url)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

print(soup.prettify())

# Remove comments from HTML
comments = soup.find_all(string=lambda text: isinstance(text, Comment))
for comment in comments:
    comment.extract()

# Save cleaned HTML to file
with open('companion_tribe_check.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())


In [ ]:
tables = soup.find_all('table', {'class': 'wikitable sortable'})

In [ ]:
second_table = tables[1]

In [ ]:
# Find the header row and get the headers
headers = [th.text.strip() for th in second_table.find('tr').find_all('th')]

In [ ]:
headers

In [ ]:
# Find the indices of the desired columns

tribe_lookup = {}

try:
    card_name_index = headers.index('Card Name')
    tribe_exclusive_index = headers.index('Tribe-exclusive?')
except ValueError as e:
    print(f"One of the required headers was not found: {e}")
else:
    # Iterate over each row (skipping the header row)
    for row in second_table.find_all('tr')[1:]:
        cells = row.find_all(['th', 'td'])
        
        if len(cells) > max(card_name_index, tribe_exclusive_index):
            card_name = cells[card_name_index].get_text(strip=True)
            tribe_name = cells[tribe_exclusive_index].get_text(strip=True)

            # Populate the dictionary directly
            tribe_lookup[card_name] = tribe_name

            print(f"Card Name: {card_name}, Tribe-exclusive?: {tribe_name}")

In [ ]:
tribe_lookup

In [ ]:
from utils.cards import TribeExclusivity

In [ ]:
for card_info in card_infos:
    tribe_name_str = tribe_lookup.get(card_info.card_name)

    if tribe_name_str:
        try:
            # Dynamically find the correct enum member
            matching_enum = next(t for t in TribeExclusivity if t.value == tribe_name_str)
            
            # Assign the enum member to the card's field
            card_info.tribe_exclusivity = matching_enum
            
        except StopIteration:
            # This handles cases where a tribe string exists but doesn't match an enum member.
            print(f"Warning: No matching TribeExclusivity enum found for '{tribe_name_str}' for card '{card_info.card_name}'")

In [ ]:
for c in card_infos:
    print(f'{c.card_name}: {c.tribe_exclusivity}')

### Test Neo4j

In [ ]:
from utils.neo4j_utils import create_neo4j_data

In [ ]:
# Put all the dictionary info of card_infos into a list
# I should make this a function really
cards_dict_data = [card.to_dict() for card in card_infos]

In [ ]:
cards_dict_data

In [ ]:
create_neo4j_data(cards_dict_data)